# <u>Series de Tiempo - ARIMA</u>

### Importando librería para predicción con series de tiempo

In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
# para instalar correctamente pmdarima se debe contar con una versión de Numpy 1.xx.xx

In [ ]:
!pip install pmdarima

In [ ]:
from pmdarima import auto_arima

In [ ]:
import warnings
warnings.filterwarnings("ignore")
def fxn():
    warnings.warn("deprecated", DeprecationWarning)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()


import itertools
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import pandas as pd
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import statsmodels as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

from math import sqrt

import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['text.color'] = 'k'
import seaborn as sns

from random import random

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_squared_log_error

### Importando Data
- Dataset: Pasajeros de aerolínea
- Unidad: Miles

In [ ]:
#Importamos la data
url = 'https://raw.githubusercontent.com/JBrianAlicorp/Business-Analytics/master/international-airline-passengers.csv'
df = pd.read_csv(url,encoding='latin1', header = None)

In [ ]:
df.columns = ['year','passengers']

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
print('Time period start: {}\nTime period end: {}'.format(df.year.min(),df.year.max()))

In [ ]:
df.shape

## 1.Preprocesamiento de data y visualización

__Convertimos el formato de fecha:__

In [ ]:
df['year'] = pd.to_datetime(df['year'], format='%Y-%m')

__Establecer índice como la columna de fecha y hora para manipulaciones más fáciles:__

---



In [ ]:
y = df.set_index('year')

In [ ]:
y.index

In [ ]:
y.isnull().sum()

In [ ]:
y.plot(figsize=(15, 6))
plt.show()

__Cajas y bigotes:__
- Los valores medianos a través de los años confirman una tendencia al alza
- Aumento constante de la propagación, o 50% medio de los datos (cuadros) con el tiempo
- Un modelo que considere la estacionalidad podría funcionar bien

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))
sns.boxplot(x = y.passengers.index.year, y = y.passengers, ax=ax)
plt.show()

### Gráficos ACF y PACF

- Repasemos los gráficos de la función de autocorrelación (ACF) y la función de autocorrelación parcial (PACF)
- Si la serie temporal es estacionaria, los gráficos ACF / PACF mostrarán una __disminución rápida de la correlación__ después de una pequeña cantidad de retraso entre los puntos.
- Estos datos no son estacionarios, ya que un gran número de observaciones anteriores están correlacionadas con valores futuros.
- Los intervalos de confianza se dibujan como un cono.
- De forma predeterminada, esto se establece en un intervalo de confianza del 95%, lo que sugiere que los valores de correlación fuera de este código son muy probablemente una correlación y no una casualidad estadística.
- La autocorrelación parcial en el retraso k es la correlación que resulta después de eliminar el efecto de cualquier correlación debido a los términos en los retrasos más cortos.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

plt.figure(figsize=(15,6))
plt.subplot(211)
plot_acf(y.passengers, ax=plt.gca(), lags = 30)
plt.subplot(212)
plot_pacf(y.passengers, ax=plt.gca(), lags = 30)
plt.show()

## Hacer series temporales estacionarias
Hay 2 razones principales detrás de la no estacionaria de un TS:

1. __Trend__ - media variable con el tiempo. Por ejemplo, en este caso vimos que, en promedio, el número de pasajeros crecía con el tiempo.
2. __Estacionalidad__ - variaciones en marcos de tiempo específicos. Por ejemplo, las personas pueden tener tendencia a comprar automóviles en un mes en particular debido a un incremento salarial o festivales.

### Transformaciones
- Podemos aplicar transformaciones que penalizan los valores más altos más que los valores más pequeños. Estos pueden tomar un registro, raíz cuadrada, raíz cúbica, etc. Tomemos una transformación de registro aquí por simplicidad:


#### Transformación logarítmica

In [ ]:
ts_log = np.log(y)
plt.figure(figsize=(15,6))
plt.plot(ts_log)
plt.show()

#### Otras posibles transformaciones:
- Transformación exponencial
- Transformación de Box Cox
- Transformación de raíz cuadrada

### Técnicas para eliminar Tendencia - Suavizado
- Alisar es tomar promedios continuos en ventanas de tiempo

#### Media móvil
- Tomamos un promedio de "k" valores consecutivos dependiendo de la frecuencia de las series de tiempo.
- Aquí podemos tomar el promedio durante el último año, es decir, los últimos 12 valores.
- Un inconveniente de este enfoque particular es que el período de tiempo debe definirse estrictamente.

In [ ]:
moving_avg = ts_log.rolling(12).mean()
plt.figure(figsize=(15,6))
plt.plot(ts_log)
plt.plot(moving_avg, color='red')
plt.show()

In [ ]:
ts_log_moving_avg_diff = ts_log.passengers - moving_avg.passengers
ts_log_moving_avg_diff.head(20)

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ts_log_moving_avg_diff.dropna(), color='red')
plt.show()

### Técnicas adicionales para eliminar la estacionalidad y la tendencia
- Las técnicas simples de reducción de tendencias discutidas antes no funcionan en todos los casos, particularmente en aquellas con alta estacionalidad.

#### Diferenciación
- En esta técnica, tomamos la diferencia de la observación en un instante particular con la del instante anterior.
- Diferenciación de primer orden en pandas

In [ ]:
ts_log_diff = ts_log.passengers - ts_log.passengers.shift()
plt.figure(figsize=(15,6))
plt.plot(ts_log_diff)
plt.show()

In [ ]:
ts_log_diff

# Predicción de series de tiempo

## Autoregresión (AR)
- El método de autorregresión (AR) modela el siguiente paso de la secuencia como una función lineal de las observaciones en los pasos de tiempo anteriores.
- __Número de términos AR (Auto-regresivos) (p):__ p es el parámetro asociado con el aspecto auto-regresivo del modelo, que incorpora valores pasados, es decir, retrasos de la variable dependiente. Por ejemplo, si p es 5, los predictores para x (t) serán x (t-1) ... .x (t-5).

In [ ]:
from statsmodels.tsa.ar_model import AutoReg, ar_select_order
from random import random

In [ ]:
ts_log_diff = ts_log_diff.dropna()

In [ ]:
plt.figure(figsize=(15,6))
plt.subplot(211)
plot_acf(ts_log_diff, ax=plt.gca(), lags = 30)
plt.subplot(212)
plot_pacf(ts_log_diff, ax=plt.gca(), lags = 30)
plt.show()

In [ ]:
# fit model
model = AutoReg(ts_log_diff, 4, old_names=False)
model_fit = model.fit()
print(model_fit.summary())

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ts_log_diff)
plt.plot(model_fit.fittedvalues, color='red')
plt.title('RSS: %.4f'% np.nansum((model_fit.fittedvalues-ts_log_diff)**2))
plt.show()

### Revirtiendo las transformaciones

__Valores ajustados o predichos:__

In [ ]:
predictions_ARIMA_diff = pd.Series(model_fit.fittedvalues, copy=True)
print (predictions_ARIMA_diff.head())

__Suma acumulativa para revertir la diferenciación:__

In [ ]:
predictions_ARIMA_diff_cumsum = predictions_ARIMA_diff.cumsum()
print (predictions_ARIMA_diff_cumsum.head())

__Agregar el valor del primer mes que se eliminó previamente al diferenciar:__

In [ ]:
predictions_ARIMA_log = pd.Series(ts_log.passengers.iloc[0], index=ts_log.index)
predictions_ARIMA_log = predictions_ARIMA_log.add(predictions_ARIMA_diff_cumsum,fill_value=0)
predictions_ARIMA_log.head()

__Tomando exponente para invertir la transformación del registro:__

In [ ]:
predictions_ARIMA = np.exp(predictions_ARIMA_log)

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(y.passengers)
plt.plot(predictions_ARIMA)
plt.title('RMSE: %.4f'% np.sqrt(np.nansum((predictions_ARIMA-y.passengers)**2)/len(y.passengers)))
plt.show()

### Métricas de puntuación de calidad de pronóstico
- __R-cuadrado__
- __Error absoluto medio__
- __Median Absolute Error__
- __Error medio cuadrado__
- __Error logarítmico medio cuadrado__
- __Error medio absoluto de porcentaje__

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_squared_log_error

__R cuadrado__, coeficiente de determinación (puede interpretarse como un porcentaje de varianza explicado por el modelo), (-inf, 1]
- sklearn.metrics.r2_score

__Mean Absolute Error__, es una métrica interpretable porque tiene la misma unidad de medida que la serie inicial, [0, + inf)
- sklearn.metrics.mean_absolute_error

In [ ]:
mean_absolute_error(y.passengers, predictions_ARIMA)

__Median Absolute Error__, nuevamente una métrica interpretable, particularmente interesante porque es robusta para los valores atípicos, [0, + inf)
- sklearn.metrics.median_absolute_error

In [ ]:
median_absolute_error(y.passengers, predictions_ARIMA)

__Mean Squared Error__, más comúnmente usado, da una penalización mayor a los grandes errores y viceversa, [0, + inf)
- sklearn.metrics.mean_squared_error


In [ ]:
mean_squared_error(y.passengers, predictions_ARIMA)**(1/2)

__Error logarítmico medio cuadrado__, prácticamente lo mismo que MSE pero inicialmente tomamos el logaritmo de la serie, como resultado también prestamos atención a pequeños errores, generalmente se usa cuando los datos tienen tendencias exponenciales, [0, + inf)

In [ ]:
mean_squared_log_error(y.passengers, predictions_ARIMA)

__Error medio porcentual absoluto__, igual que MAE pero porcentaje, - muy conveniente cuando desea explicar la calidad del modelo a su gerencia, [0, + inf),
- no implementado en sklearn

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
mean_absolute_percentage_error(y.passengers, predictions_ARIMA)

__Función para evaluar el pronóstico utilizando las métricas anteriores:__

In [ ]:
def evaluate_forecast(y,pred):
    results = pd.DataFrame({'r2_score':r2_score(y, pred),
                           }, index=[0])
    results['mean_absolute_error'] = mean_absolute_error(y, pred)
    results['median_absolute_error'] = median_absolute_error(y, pred)
    results['mse'] = mean_squared_error(y, pred)
    results['msle'] = mean_squared_log_error(y, pred)
    results['mape'] = mean_absolute_percentage_error(y, pred)
    results['rmse'] = np.sqrt(results['mse'])
    return results

In [ ]:
evaluate_forecast(y.passengers, predictions_ARIMA)

- RMSE tiene el beneficio de penalizar más errores grandes, por lo que puede ser más apropiado en algunos casos.

- Desde el punto de vista de la interpretación, MAE es claramente el ganador. RMSE no describe solo el error promedio y tiene otras implicaciones que son más difíciles de descifrar y comprender.

- Por otro lado, una ventaja distintiva de RMSE sobre MAE es que RMSE evita el uso de tomar el valor absoluto, lo que no es deseable en muchos cálculos matemáticos.

## Media móvil (MA)

- __Número de términos MA (promedio móvil) (q):__ q es el tamaño de la ventana de parte del promedio móvil del modelo, es decir, errores de pronóstico rezagados en la ecuación de predicción. Por ejemplo, si q es 5, los predictores para x (t) serán e (t-1)… .e (t-5) donde e (i) es la diferencia entre el promedio móvil en el valor instantáneo y real.

In [ ]:
# MA example
from statsmodels.tsa.arima.model import ARIMA
from random import random

# fit model
model = ARIMA(ts_log_diff, order=(0, 0, 4))
model_fit = model.fit()

In [ ]:
model_fit.summary()

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ts_log_diff)
plt.plot(model_fit.fittedvalues, color='red')
plt.title('RSS: %.4f'% np.nansum((model_fit.fittedvalues-ts_log_diff)**2))
plt.show()

In [ ]:
predictions_ARIMA_diff = pd.Series(model_fit.fittedvalues, copy=True)
print (predictions_ARIMA_diff.head())

In [ ]:
predictions_ARIMA_diff_cumsum = predictions_ARIMA_diff.cumsum()
print (predictions_ARIMA_diff_cumsum.head())

In [ ]:
predictions_ARIMA_log = pd.Series(ts_log.passengers.iloc[0], index=ts_log.index)
predictions_ARIMA_log = predictions_ARIMA_log.add(predictions_ARIMA_diff_cumsum,fill_value=0)
predictions_ARIMA_log.head()

In [ ]:
predictions_ARIMA = np.exp(predictions_ARIMA_log)

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(y.passengers)
plt.plot(predictions_ARIMA)
plt.title('RMSE: %.4f'% np.sqrt(np.nansum((predictions_ARIMA-y.passengers)**2)/len(y.passengers)))
plt.show()

In [ ]:
evaluate_forecast(y.passengers, predictions_ARIMA)

## Media móvil autorregresiva (ARMA)

- __Número de términos AR (Auto-regresivos) (p):__ p es el parámetro asociado con el aspecto auto-regresivo del modelo, que incorpora valores pasados, es decir, retrasos de la variable dependiente. Por ejemplo, si p es 5, los predictores para x (t) serán x (t-1) ... .x (t-5).
- __Número de términos MA (promedio móvil) (q):__ q es el tamaño de la ventana de parte del promedio móvil del modelo, es decir, errores de pronóstico rezagados en la ecuación de predicción. Por ejemplo, si q es 5, los predictores para x (t) serán e (t-1)… .e (t-5) donde e (i) es la diferencia entre el promedio móvil en el valor instantáneo y real.


In [ ]:
# ARMA example
from statsmodels.tsa.arima.model import ARIMA
from random import random

# fit model
model = ARIMA(ts_log_diff, order=(1,0,4))
model_fit = model.fit()

In [ ]:
model_fit.summary()

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ts_log_diff)
plt.plot(model_fit.fittedvalues, color='red')
plt.title('RSS: %.4f'% np.nansum((model_fit.fittedvalues-ts_log_diff)**2))
plt.show()

In [ ]:
predictions_ARIMA_diff = pd.Series(model_fit.fittedvalues, copy=True)
print (predictions_ARIMA_diff.head())

In [ ]:
predictions_ARIMA_diff_cumsum = predictions_ARIMA_diff.cumsum()
print (predictions_ARIMA_diff_cumsum.head())

In [ ]:
predictions_ARIMA_log = pd.Series(ts_log.passengers.iloc[0], index=ts_log.index)
predictions_ARIMA_log = predictions_ARIMA_log.add(predictions_ARIMA_diff_cumsum,fill_value=0)
predictions_ARIMA_log.head()

In [ ]:
predictions_ARIMA = np.exp(predictions_ARIMA_log)

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(y.passengers)
plt.plot(predictions_ARIMA)
plt.title('RMSE: %.4f'% np.sqrt(np.nansum((predictions_ARIMA-y.passengers)**2)/len(y.passengers)))
plt.show()

In [ ]:
evaluate_forecast(y.passengers, predictions_ARIMA)

## Media móvil integrada autorregresiva (ARIMA)
En un modelo ARIMA hay 3 parámetros que se utilizan para ayudar a modelar los aspectos principales de una serie de tiempo: estacionalidad, tendencia y ruido. Estos parámetros están etiquetados como p, d y q.

- __Número de términos AR (Auto-regresivos) (p):__ p es el parámetro asociado con el aspecto auto-regresivo del modelo, que incorpora valores pasados, es decir, retrasos de la variable dependiente. Por ejemplo, si p es 5, los predictores para x (t) serán x (t-1) ... .x (t-5).
- __Número de diferencias (d):__ d es el parámetro asociado con la parte integrada del modelo, que afecta la cantidad de diferencia que se aplica a una serie de tiempo.
- __Número de términos MA (promedio móvil) (q):__ q es el tamaño de la ventana de parte del promedio móvil del modelo, es decir, errores de pronóstico rezagados en la ecuación de predicción. Por ejemplo, si q es 5, los predictores para x (t) serán e (t-1)… .e (t-5) donde e (i) es la diferencia entre el promedio móvil en el valor instantáneo y real.

<br> __Observaciones de EDA en la serie temporal:__
- La no estacionariedad implica que se requiere al menos un nivel de diferencia (d) en ARIMA
- [El siguiente paso es seleccionar los valores de retraso para los parámetros de Autoregresión (AR) y Media móvil (MA), p y q respectivamente, utilizando gráficos PACF, ACF]

[Ajuste de los parámetros de ARIMA] (https://machinelearningmastery.com/tune-arima-parameters-python/)


Nota: Un problema con ARIMA es que no admite datos estacionales. Esa es una serie de tiempo con un ciclo repetitivo. ARIMA espera datos que no sean estacionales o que eliminen el componente estacional, p. ajustado estacionalmente mediante métodos como la diferenciación estacional.

In [ ]:
ts = y.passengers - y.passengers.shift()
ts.dropna(inplace=True)

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ts)
plt.show()

__Gráficos ACF y PACF después de diferenciar:__
- Los intervalos de confianza se dibujan como un cono.
- De forma predeterminada, esto se establece en un intervalo de confianza del 95%, lo que sugiere que los valores de correlación fuera de este código son muy probablemente una correlación y no una casualidad estadística.
- Proceso AR (1): tiene un ACF que se reduce y el PACF se corta en el retraso = 1
- Proceso AR (2): tiene un ACF que se reduce y el PACF se corta en el retraso = 2
- Proceso MA (1): tiene un corte de ACF en el retraso = 1
- Proceso MA (2) - tiene un corte de ACF en el retraso = 2


In [ ]:
plt.figure(figsize=(15,6))
plt.subplot(211)
plot_acf(ts, ax=plt.gca(),lags=30)
plt.subplot(212)
plot_pacf(ts, ax=plt.gca(),lags=30)
plt.show()

## Interpretando diagramas ACF


ACF Shape | Modelo indicado |
- | - |
Exponencial, decayendo a cero | Modelo autorregresivo. Utilice el gráfico de autocorrelación parcial para identificar el orden del modelo autorregresivo |
Alternancia positiva y negativa, decayendo a cero Modelo autorregresivo.  | Use el gráfico de autocorrelación parcial para ayudar a identificar el orden. |
Uno o más picos, el resto son esencialmente cero | Modelo de media móvil, orden identificado por donde la trama se convierte en cero.  |
Decaimiento, comenzando después de algunos retrasos | Modelo mixto autorregresivo y de media móvil (ARMA). |
Todo cero o cerca de cero | Los datos son esencialmente aleatorios.  |
Valores altos a intervalos fijos | Incluye término estacional autorregresivo.  |
Sin descomposición a cero | La serie no es estacionaria |

In [ ]:
# tranformación logaritmo
ylog = np.log(y)

#divide into train and validation set
train = ylog[:int(0.90*(len(ylog)))]
valid = ylog[int(0.90*(len(ylog))):]

#plotting the data
plt.figure(figsize=(15,6))
train['passengers'].plot()
valid['passengers'].plot()

### Auto ARIMA

In [ ]:
from pmdarima import auto_arima

In [ ]:
#building the model
from pmdarima import auto_arima
model = auto_arima(train, trace=True, error_action='ignore', suppress_warnings=True)
model.fit(train)

In [ ]:
forecast = model.predict(n_periods=len(valid))
forecast = pd.DataFrame(forecast,index = valid.index,columns=['Prediction'])

plt.figure(figsize=(15,6))
#plot the predictions for validation set
plt.plot(ylog.passengers, label='Train')
plt.plot(valid, label='Valid')
plt.plot(forecast, label='Prediction')
plt.show()

In [ ]:
evaluate_forecast(valid, forecast)

In [ ]:
evaluate_forecast(np.exp(valid), np.exp(forecast))

## Promedio móvil integrado estacional autorregresivo (SARIMA)
El promedio móvil integrado estacional autorregresivo, SARIMA o ARIMA estacional, es una extensión de ARIMA que admite explícitamente datos de series temporales univariantes con un componente estacional.

Agrega tres nuevos hiperparámetros para especificar la autorregresión (AR), la diferenciación (I) y el promedio móvil (MA) para el componente estacional de la serie, así como un parámetro adicional para el período de estacionalidad.

__Elementos de tendencia:__

Hay tres elementos de tendencia que requieren configuración. Son los mismos que el modelo ARIMA, específicamente:

- p: orden de tendencia de autorregresión.
- d: orden de diferencia de tendencia.
- q: tendencia de la media móvil.

__Elementos estacionales:__

Hay cuatro elementos estacionales que no forman parte de ARIMA que deben configurarse; son:

- P: orden autorregresivo estacional.
- D: orden de diferencia estacional.
- Q: orden de media móvil estacional.
- m: el número de pasos de tiempo para un solo período estacional. Por ejemplo, una S de 12 para datos mensuales sugiere un ciclo estacional anual.

__Notación SARIMA:__
SARIMA (p, d, q) (P, D, Q, m)

In [ ]:
#divide into train and validation set
train = ylog[:int(0.90*(len(ylog)))]
valid = ylog[int(0.90*(len(ylog))):]

#plotting the data
plt.figure(figsize=(15,6))
train['passengers'].plot()
valid['passengers'].plot()

In [ ]:
# SARIMA example
from statsmodels.tsa.statespace.sarimax import SARIMAX

# fit model
model = SARIMAX(train, order=(2, 1, 2), seasonal_order=(1, 1, 2, 12))
model_fit = model.fit(disp=False)

In [ ]:
start_index = valid.index.min()
end_index = valid.index.max()

#Predictions
predictions = model_fit.predict(start=start_index, end=end_index)

In [ ]:
# report performance
mse = mean_squared_error(ylog[start_index:end_index], predictions)
rmse = sqrt(mse)
print('RMSE: {}, MSE:{}'.format(rmse,mse))

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(ylog)
plt.plot(predictions)
plt.title('RMSE: %.4f'% rmse)
plt.show()

In [ ]:
evaluate_forecast(ylog[start_index:end_index], predictions)

In [ ]:
evaluate_forecast(np.exp(ylog[start_index:end_index]), np.exp(predictions))

### Auto - SARIMA

[auto_arima documentation for selecting best model](https://www.alkaline-ml.com/pmdarima/tips_and_tricks.html)

In [ ]:
#building the model
from pmdarima import auto_arima
model = auto_arima(train, trace=True, error_action='ignore', suppress_warnings=True, seasonal=True, m=12, stepwise=True)
model.fit(train)

In [ ]:
start_index = valid.index.min()
end_index = valid.index.max()

#Predictions
pred = model.predict()

In [ ]:
pred = model.predict(n_periods=len(valid))
pred = pd.DataFrame(pred,index = valid.index,columns=['Prediction'])

In [ ]:
forecast = model.predict(n_periods=len(valid))
forecast = pd.DataFrame(forecast,index = valid.index,columns=['Prediction'])

plt.figure(figsize=(15,6))
#plot the predictions for validation set
plt.plot(ylog.passengers, label='Train')
#plt.plot(valid, label='Valid')
plt.plot(forecast, label='Prediction')
plt.show()

In [ ]:
evaluate_forecast(ylog[start_index:end_index], forecast)

In [ ]:
evaluate_forecast(np.exp(ylog[start_index:end_index]), np.exp(forecast))

__Diagnóstico del modelo:__
- Nuestra principal preocupación es garantizar que los residuos de nuestro modelo no estén correlacionados y normalmente se distribuyan con media cero.
- Si el modelo estacional ARIMA no satisface estas propiedades, es una buena indicación de que puede mejorarse aún más.

El diagnóstico del modelo sugiere que el modelo residual se distribuye normalmente en función de lo siguiente:

- En la gráfica superior derecha, la línea roja de KDE sigue de cerca con la línea N (0,1). Donde, N (0,1) es la notación estándar para una distribución normal con media 0 y desviación estándar de 1. Esta es una buena indicación de que los residuos se distribuyen normalmente.
- La gráfica qq en la parte inferior izquierda muestra que la distribución ordenada de los residuos (puntos azules) sigue la tendencia lineal de las muestras tomadas de una distribución normal estándar. Nuevamente, esta es una fuerte indicación de que los residuos se distribuyen normalmente.
- Los residuos a lo largo del tiempo (gráfico superior izquierdo) no muestran ninguna estacionalidad obvia y parecen ser ruido blanco.
- Esto se confirma mediante el gráfico de autocorrelación (es decir, correlograma) en la parte inferior derecha, que muestra que los residuos de series temporales tienen una baja correlación con versiones rezagadas de sí mismo.

In [ ]:
model.plot_diagnostics(figsize=(16, 8))
plt.show()

## Transformación BoxCox

y = (x**lmbda - 1) / lmbda,  for lmbda != 0

log(x),                  for lmbda = 0

In [ ]:
import scipy.stats as ss
yt = ss.boxcox(y.passengers)

In [ ]:
yt[1]  #lambda óptimo

In [ ]:
lam = yt[1]

In [ ]:
yt[0]

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(yt[0])
plt.show()

In [ ]:
yt = pd.DataFrame({'passengers':yt[0]})

In [ ]:
#divide into train and validation set
train = yt[:int(0.9*(len(yt)))]
valid = yt[int(0.9*(len(yt))):]

#plotting the data
plt.figure(figsize=(15,6))
train['passengers'].plot()
valid['passengers'].plot()

In [ ]:
#building the model
from pmdarima import auto_arima
model = auto_arima(train, trace=True, error_action='ignore', suppress_warnings=True, seasonal=True, m=12, stepwise=True)
model.fit(train)

In [ ]:
start_index = valid.index.min()
end_index = valid.index.max()

#Predictions
pred = model.predict()

In [ ]:
pred = model.predict(n_periods=len(valid))
pred = pd.DataFrame(pred,index = valid.index,columns=['Prediction'])

In [ ]:
forecast = model.predict(n_periods=len(valid))
forecast = pd.DataFrame(forecast,index = valid.index,columns=['Prediction'])

plt.figure(figsize=(15,6))
#plot the predictions for validation set
plt.plot(yt.passengers, label='Train')
#plt.plot(valid, label='Valid')
plt.plot(forecast, label='Prediction')
plt.show()

In [ ]:
evaluate_forecast(yt.passengers[-15:], forecast)

In [ ]:
lam

In [ ]:
evaluate_forecast((yt.passengers[-15:]*lam+1)**(1/lam), (forecast*lam+1)**(1/lam))